# Conditional Probability, Bayes Rule, And Sampling

Official MA1001B alignment: 1.3 conditional probability and Bayes' rule; 1.4 random sampling.


## How To Use This Lesson

Read the explanation cells before running the code. Run each code cell in order. When a checkpoint appears, stop and write your answer before continuing. The goal is not only to obtain output; the goal is to justify a decision from data.


## Learning Goals

- Explain the statistical idea in words.
- Implement the idea in Python with readable code.
- Interpret the result as evidence for a decision.
- State at least one assumption or limitation.


## Decision Scenario

A safety analyst is asked to communicate risk from passenger data. The analyst must distinguish P(survived | group) from P(group | survived), because they answer different questions.


## Conceptual Explanation

Conditional probability restricts the reference group. P(A | B) means the probability of A among cases where B is true. Bayes' rule lets us reverse a conditional probability when we also know the base rates. In data science, many communication errors come from switching the condition and the outcome.


## Mathematical Anchor

P(A | B) = P(A intersection B) / P(B). Bayes' rule: P(A | B) = P(B | A) P(A) / P(B).


## Data And Workflow Notes

Uses Titanic if `data/raw/titanic/train.csv` exists; otherwise uses a simulated table with the same kind of variables.


## Python Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)


## Worked Example


In [ ]:
path = Path("data/raw/titanic/train.csv")
if path.exists():
    passengers = pd.read_csv(path).rename(
        columns={"Survived": "survived", "Sex": "sex", "Pclass": "pclass"}
    )
else:
    print(f"Missing {path}. Download Titanic from https://www.kaggle.com/c/titanic")
    passengers = pd.DataFrame({
        "survived": rng.binomial(1, 0.38, 891),
        "sex": rng.choice(["female", "male"], 891, p=[0.36, 0.64]),
        "pclass": rng.choice([1, 2, 3], 891, p=[0.24, 0.21, 0.55]),
    })

passengers[["survived", "sex", "pclass"]].head()


In [ ]:
survived = passengers["survived"].eq(1)
female = passengers["sex"].eq("female")

pd.Series({
    "P(survived)": survived.mean(),
    "P(female)": female.mean(),
    "P(survived | female)": survived[female].mean(),
    "P(female | survived)": female[survived].mean(),
}).round(3)


## From Calculation To Evidence


In [ ]:
conditional_table = (
    passengers
    .groupby(["sex", "pclass"])["survived"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "survival_rate"})
    .sort_values("survival_rate", ascending=False)
)
conditional_table


In [ ]:
sample_estimates = []
for seed in range(30):
    sampled = passengers.sample(120, random_state=seed)
    sample_estimates.append(sampled.loc[sampled["sex"].eq("female"), "survived"].mean())

pd.Series(sample_estimates).describe()


## Guided Checkpoint

Write two sentences: one using P(survived | female) correctly and one using P(female | survived) correctly.


## Common Mistakes

- Reversing the condition and the outcome.
- Ignoring base rates when interpreting Bayes' rule.
- Treating historical associations as proof of a causal survival mechanism.


## Independent Practice

Choose another subgroup, estimate its survival probability, and compare the full-data estimate with repeated samples of size 120.


## Interpretation Template

Use this structure for your written answer:

1. The decision question is ...
2. The statistical evidence is ...
3. The uncertainty or limitation is ...
4. Therefore, I recommend ... because ...


## Exit Ticket

What information is lost when a probability is reported without its conditioning group?
